In [1]:
import numpy as np
import pandas as pd

In [ ]:
def load_data(file_path):
    """
    Load data from a CSV file and return a DataFrame.
    
    Parameters:
    file_path (str): Path to the CSV file.
    
    Returns:
    pd.DataFrame: DataFrame containing the loaded data.
    """
    try:
        data = pd.read_csv(file_path, header=None)
        return data
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

In [18]:
# load the training and test datasets
train_path = 'train.csv'
test_path = 'test.csv'
train = load_data(train_path)
test = load_data(test_path)

C:\Users\Bernd\AppData\Local\Temp\ipykernel_35328\660518726.py:12: DtypeWarning: Columns (0,2,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file_path, header=None)
C:\Users\Bernd\AppData\Local\Temp\ipykernel_35328\660518726.py:12: DtypeWarning: Columns (0,2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file_path, header=None)


### Train Data Preprocessing

In [19]:
print(f"Training data shape: {train.shape}")
train.head()

Training data shape: (750001, 9)


,0,1,2,3,4,5,6,7,8
0,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
1,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
2,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
3,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
4,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0


In [ ]:
# check for missing values
if train is not None:
    missing_values = train.isnull().sum()

    # Handle missing values in training data
    train_df = train.fillna(train.mean())

    print(f"Missing values in training data:\n{missing_values}")

Missing values in training data:
0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
dtype: int64


In [21]:
# check for duplicate values
if train is not None:
    duplicate_rows = train.duplicated().sum()
    print(f"Number of duplicate rows in training data: {duplicate_rows}")

Number of duplicate rows in training data: 0


In [22]:
# check the distribution of the sex column
train[1].value_counts()

female    375721
male      374279
Sex            1
Name: 1, dtype: int64

In [23]:
# set the first row as column names
train.columns = train.iloc[0]
train = train[1:]  # remove the first row

# reset the index
train.reset_index(drop=True)

# set 'id' as the index
train.set_index('id')

# ensure data types are correct
numeric_cols = ['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp', 'Calories']
train[numeric_cols] = train[numeric_cols].apply(pd.to_numeric, errors='coerce')

train.head()

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
1,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
2,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
3,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
4,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
5,4,female,38,166.0,61.0,25.0,102.0,40.6,146.0


In [24]:
train['Sex'].value_counts()

female    375721
male      374279
Name: Sex, dtype: int64

In [25]:
# Encode categorical variables
train['Sex'] = train['Sex'].map({'male': 1, 'female': 2, 'other': 3})

# Check for any remaining missing values after encoding
missing_values_after_encoding = train.isnull().sum()
print(f"Missing values after encoding:\n{missing_values_after_encoding}")

Missing values after encoding:
0
id            0
Sex           0
Age           0
Height        0
Weight        0
Duration      0
Heart_Rate    0
Body_Temp     0
Calories      0
dtype: int64


In [ ]:
# Transform target to log(Calories + 1) to avoid log(0) and align with RMSLE
train['log_Calories'] = np.log1p(train['Calories'])

### Model Development

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Features (X) and target (y)
X = train[['Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp']]
y = train['log_Calories']

In [ ]:
# Handle missing values (if any, though your sample looks clean)
X = X.fillna(X.mean())
# y = y.fillna(y.mean())

In [29]:
# Split the data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [30]:
# Initialize and train the linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [31]:
# Make predictions on the test set
y_pred = model.predict(X_test)

In [32]:
# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [33]:
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R² Score: {r2:.2f}")
print("\nFeature Coefficients:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"{feature}: {coef:.2f}")

Mean Squared Error (MSE): 122.30
R² Score: 0.97

Feature Coefficients:
Sex: 1.69
Age: 0.53
Height: -0.15
Weight: 0.27
Duration: 6.76
Heart_Rate: 1.95
Body_Temp: -18.20


### Submission

In [34]:
test_df = pd.read_csv('test.csv', index_col='id')
test_df.head()

,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
id,,,,,,,
750000,male,45,177.0,81.0,7.0,87.0,39.8
750001,male,26,200.0,97.0,20.0,101.0,40.5
750002,female,29,188.0,85.0,16.0,102.0,40.4
750003,female,39,172.0,73.0,20.0,107.0,40.6
750004,female,30,173.0,67.0,16.0,94.0,40.5


In [36]:
# Encode 'Sex' column in test data
test_df['Sex'] = test_df['Sex'].map({'male': 1, 'female': 2})

# Ensure numeric columns in test data
test_df[numeric_cols[:-1]] = test_df[numeric_cols[:-1]].apply(pd.to_numeric, errors='coerce')

# Handle missing values in test data
test_df = test_df.fillna(test_df.mean())

# Features for prediction
X_test = test_df[['Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp']]

In [37]:
# Make predictions
predictions = model.predict(X_test)

In [39]:
# create submission dataframe
submission_df = pd.DataFrame({
    'id': test_df.index,
    'Calories': predictions
})

In [40]:
# Ensure 'Calories' is rounded to 3 decimal places to match sample_submission.csv
submission_df['Calories'] = submission_df['Calories'].round(3)

In [41]:
# Save to CSV
submission_df.to_csv('submission.csv', index=False)

In [42]:
print("Submission file 'submission.csv' created successfully!")

Submission file 'submission.csv' created successfully!
